# Train the same forecasting model on a free Colab GPU

This notebook runs the project CLI and saves resumable checkpoints to your Drive. Free GPU access and session lengths vary; a session may end before training completes. The browser visualizer runs locally after downloading the resulting checkpoint. No results are included in this notebook.

Before running: choose a GPU runtime, upload or check out this repository to `/content/adp`, and upload a ZIP of the selected real AV2 raw subset to Drive. The ZIP contents must have `manifest.json` at their root, alongside the split directories. Use the same manifest as the local run. This project uses Python 3.12; select a compatible runtime if the preflight reports another version.

In [ ]:
import importlib.util
import json
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/adp')  # Change this to your uploaded repository.
assert (REPO_DIR / 'pyproject.toml').is_file(), 'Upload/check out the repository and set REPO_DIR first.'
print('Python:', sys.version)
assert sys.version_info[:2] == (3, 12), 'Use a Python 3.12 runtime, matching this project.'
assert importlib.util.find_spec('torch'), 'Choose a GPU runtime with PyTorch installed.'
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 7), 'PyTorch 2.7 or later is required.'
assert torch.cuda.is_available(), 'No GPU is available. Check the runtime accelerator and free quota.'
x = torch.randn(32, 32, device='cuda', requires_grad=True)
(x @ x.T).square().mean().backward()
torch.cuda.synchronize()
print('GPU:', torch.cuda.get_device_name(0))
print('Free GPU memory:', torch.cuda.mem_get_info()[0])

Install the project into this runtime, keeping the compatible PyTorch build already supplied by Colab. Do not copy the laptop's CUDA-wheel command into every cloud runtime: its GPU may have a different architecture.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-e', str(REPO_DIR)], check=True)
from adp.training import gpu_smoke_test
config_path = REPO_DIR / 'config.json'
config = json.loads(config_path.read_text())
print(json.dumps(gpu_smoke_test(config=config), indent=2))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/autonomous-driving-prediction')
RAW_ARCHIVE = DRIVE_PROJECT / 'raw_subset.zip'
RUN_DIR = DRIVE_PROJECT / 'runs' / 'joint-mixture-v1'
LOCAL_DATA = Path('/content/adp_data')
RAW_DIR = LOCAL_DATA / 'raw'
CACHE_DIR = LOCAL_DATA / 'cache'
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_ARCHIVE.is_file(), f'Upload the selected raw subset archive to {RAW_ARCHIVE}'

Copy and extract the archive onto the runtime's local disk. Training from thousands of tiny files on mounted Drive is slow and can exhaust its I/O quotas. The raw data and cache here are disposable; checkpoints stay on Drive.

In [ ]:
import shutil
import zipfile

LOCAL_DATA.mkdir(parents=True, exist_ok=True)
local_archive = LOCAL_DATA / 'raw_subset.zip'
shutil.copyfile(RAW_ARCHIVE, local_archive)
RAW_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_archive) as archive:
    root = RAW_DIR.resolve()
    for member in archive.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(RAW_DIR)
assert (RAW_DIR / 'manifest.json').is_file(), 'Archive contents must have manifest.json at their root.'
subprocess.run([sys.executable, '-m', 'adp', 'preprocess', '--raw-dir', str(RAW_DIR), '--manifest', str(RAW_DIR / 'manifest.json'), '--cache-dir', str(CACHE_DIR)], cwd=REPO_DIR, check=True)

Train or resume. `--epochs` is the requested total epoch count, including completed epochs. The trainer verifies the exact configuration and manifest before resuming, writes `last.pt` each epoch and every five minutes, and updates `best.pt` on validation improvement. Timed checkpoints from an incomplete epoch restart that epoch from its beginning. Keeping the same run directory also preserves the prior best checkpoint.

For the first cloud run, use a new run name above. To test the pipeline, set `TOTAL_EPOCHS = 1`; then increase it for a longer experiment. Checkpoints contain measured losses, not benchmark claims. Your laptop's batch and accumulation settings also work here; adjust configuration before starting a new run if needed.

In [ ]:
TOTAL_EPOCHS = 20
last_checkpoint = RUN_DIR / 'last.pt'
command = [sys.executable, '-m', 'adp', 'train', '--cache-dir', str(CACHE_DIR), '--output-dir', str(RUN_DIR), '--config', str(config_path), '--epochs', str(TOTAL_EPOCHS), '--device', 'cuda']
if last_checkpoint.is_file():
    command += ['--resume', str(last_checkpoint)]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

Fit calibration only on the reserved calibration split, using the best validation checkpoint. The JSON explicitly marks calibration as unfitted when no eligible targets or only one risk class is present. Keep the separate evaluation split for final reporting.

In [ ]:
best_checkpoint = RUN_DIR / 'best.pt'
assert best_checkpoint.is_file(), 'Complete at least one training epoch first.'
subprocess.run([sys.executable, '-m', 'adp', 'calibrate', '--checkpoint', str(best_checkpoint), '--cache-dir', str(CACHE_DIR), '--output-dir', str(RUN_DIR), '--device', 'cuda'], cwd=REPO_DIR, check=True)
print((RUN_DIR / 'calibration.json').read_text())

Evaluate the reserved holdout only after selecting the checkpoint. The same CLI writes JSON records, scenario-bootstrap intervals and reliability figures to Drive. Compare against constant velocity; sparse positive-event counts still mean insufficient calibration evidence.

In [ ]:
subprocess.run([sys.executable, '-m', 'adp', 'evaluate', '--checkpoint', str(best_checkpoint), '--calibration', str(RUN_DIR / 'calibration.json'), '--cache-dir', str(CACHE_DIR), '--output-dir', str(RUN_DIR / 'evaluation'), '--device', 'cuda'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'adp', 'evaluate', '--cache-dir', str(CACHE_DIR), '--output-dir', str(RUN_DIR / 'baseline'), '--device', 'cuda'], cwd=REPO_DIR, check=True)

Download `best.pt`, `last.pt`, `calibration.json`, `history.json`, `run.json`, and the evaluation folders from the run folder in Drive. Use `best.pt` with the local evaluator and browser visualizer. Resume with `last.pt`; keep the original source, configuration, and manifest. Stop the GPU runtime when done. Free resource availability is described in the [Colab FAQ](https://research.google.com/colaboratory/faq.html).